# Cuaderno de reproducción — ARTISYNC

Este cuaderno recalcula, **desde los archivos crudos ya versionados en el repositorio** (sin
volver a ejecutar `k6`, Lighthouse ni la suite de tests), las cifras que ya están publicadas en
los reportes de `docs/mediciones/`. Es la materialización directa de la regla de reproducibilidad
de la guía de entrega (§4.1) y cierra **OBS-R1-05** (catálogo de observaciones, eje
Reproducibilidad, criterio R1).

Cada sección reutiliza la lógica de un script `.py` ya existente en el proyecto (importado como
módulo, no reimplementado) y termina comparando el resultado recalculado contra el número ya
publicado en el reporte correspondiente.

**Requisito:** ejecutar este notebook con working directory = `docs/mediciones/` (el directorio
donde vive este archivo). `jupyter nbconvert --execute` lo hace así por defecto; si se abre en
Jupyter Lab/Notebook manualmente, confirmar que el cwd del kernel sea esta carpeta.

Fuentes recalculadas:
1. **Cobertura** — `docs/mediciones/jacoco/html/jacoco.csv` (lógica de
   `artisync/Backend/analyze_coverage.py`).
2. **Percentiles + test inferencial** — NDJSON de k6 en `docs/mediciones/perf/`
   (`docs/mediciones/perf/analisis-inferencial.py`).
3. **SUS** — `docs/mediciones/sus/sus-raw.csv`
   (`docs/mediciones/sus/analisis-sus.py` + `bootstrap-sus.py`).
4. **Lighthouse** — corrida final canónica `lhci-20260904-1545-*` en
   `docs/mediciones/lighthouse/`.


## 1. Cobertura (JaCoCo)

Recalcula GLOBAL / SERVICIOS / CONTROLADORES a partir de
`docs/mediciones/jacoco/html/jacoco.csv`, con la misma lógica de agregación que
`artisync/Backend/analyze_coverage.py` (ese script apunta por defecto a
`target/site/jacoco/jacoco.csv`, una ruta de build de Maven que no existe en un clon limpio sin
compilar; aquí se aplica la misma lógica sobre el CSV versionado).

In [1]:
import csv
from collections import defaultdict
from pathlib import Path

JACOCO_CSV = Path("jacoco/html/jacoco.csv")

with JACOCO_CSV.open(encoding="utf-8") as f:
    reader = csv.DictReader(f)
    stats_by_package = defaultdict(lambda: {
        "line_missed": 0, "line_covered": 0, "branch_missed": 0, "branch_covered": 0
    })
    classes = []
    for row in reader:
        pkg = row["PACKAGE"]
        lm, lc = int(row["LINE_MISSED"]), int(row["LINE_COVERED"])
        bm, bc = int(row["BRANCH_MISSED"]), int(row["BRANCH_COVERED"])
        stats_by_package[pkg]["line_missed"] += lm
        stats_by_package[pkg]["line_covered"] += lc
        stats_by_package[pkg]["branch_missed"] += bm
        stats_by_package[pkg]["branch_covered"] += bc
        classes.append({"pkg": pkg, "cls": row["CLASS"], "lm": lm, "lc": lc, "bm": bm, "bc": bc})


def resumen(nombre, filtro):
    lc = sum(c["lc"] for c in classes if filtro(c["pkg"]))
    lm = sum(c["lm"] for c in classes if filtro(c["pkg"]))
    bc = sum(c["bc"] for c in classes if filtro(c["pkg"]))
    bm = sum(c["bm"] for c in classes if filtro(c["pkg"]))
    lt, bt = lc + lm, bc + bm
    lp = (lc / lt * 100) if lt else 100.0
    bp = (bc / bt * 100) if bt else 100.0
    print(f"{nombre}: Lineas {lc}/{lt} ({lp:.2f}%) | Ramas {bc}/{bt} ({bp:.2f}%)")
    return lp, bp


print(f"Archivo: {JACOCO_CSV} -- {len(classes)} clases, {len(stats_by_package)} paquetes")
print()
global_lp, global_bp = resumen("GLOBAL", lambda pkg: True)
serv_lp, serv_bp = resumen("SERVICIOS", lambda pkg: "service" in pkg.lower())
ctrl_lp, ctrl_bp = resumen("CONTROLADORES", lambda pkg: "controller" in pkg.lower())

print()
print("--- TOP 10 CONTROLADORES CON PEOR COBERTURA (lineas faltantes) ---")
controladores = sorted(
    (c for c in classes if "controller" in c["pkg"].lower()),
    key=lambda c: c["lm"], reverse=True,
)
for c in controladores[:10]:
    print(f"{c['cls']}: faltan {c['lm']} lineas, cubiertas {c['lc']}")


Archivo: jacoco\html\jacoco.csv -- 231 clases, 48 paquetes

GLOBAL: Lineas 6823/7346 (92.88%) | Ramas 1795/2191 (81.93%)
SERVICIOS: Lineas 5247/5641 (93.02%) | Ramas 1385/1699 (81.52%)
CONTROLADORES: Lineas 426/466 (91.42%) | Ramas 73/82 (89.02%)

--- TOP 10 CONTROLADORES CON PEOR COBERTURA (lineas faltantes) ---
OfferingController: faltan 13 lineas, cubiertas 13
RolePermissionController: faltan 4 lineas, cubiertas 5
AdminUserController: faltan 4 lineas, cubiertas 10
CategoryController: faltan 4 lineas, cubiertas 9
TwoFactorController: faltan 3 lineas, cubiertas 0
CountryController: faltan 3 lineas, cubiertas 3
PayPalWebhookController: faltan 3 lineas, cubiertas 1
AdminViolationController: faltan 3 lineas, cubiertas 0
UserController: faltan 2 lineas, cubiertas 17
PortfolioItemController: faltan 1 lineas, cubiertas 15


**Comparar contra:** el informe de cobertura publicado (busca en el repositorio la cifra
GLOBAL/SERVICIOS/CONTROLADORES vigente, p. ej. en el bloque de calidad del informe final o en la
bitácora de la Fase correspondiente) — deben coincidir dígito a dígito porque es el mismo CSV
versionado y la misma fórmula de agregación.

## 2. Percentiles + test inferencial (k6)

Reutiliza `docs/mediciones/perf/analisis-inferencial.py` importado como módulo (mismo patrón que
usa `docs/mediciones/sus/bootstrap-sus.py` para reusar `analisis-sus.py`): no se reimplementa la
fórmula de percentil, Welch t-test, d de Cohen, Mann-Whitney U ni A12 de Vargha-Delaney, se llama
directamente a las funciones ya auditadas. Requiere `scipy` (ver
`docs/mediciones/requirements.txt`).

In [2]:
import importlib.util
from pathlib import Path

def cargar_modulo(ruta, nombre):
    spec = importlib.util.spec_from_file_location(nombre, ruta)
    modulo = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modulo)
    return modulo

perf = cargar_modulo(Path("perf/analisis-inferencial.py"), "analisis_inferencial")

# Las rutas de COMPARACIONES en el modulo son relativas al propio archivo (perf/), asi que
# ya apuntan correctamente a docs/mediciones/perf/*.json sin importar el cwd de este notebook.
resultados = []
for comp in perf.COMPARACIONES:
    caliente, statuses_c = perf.cargar_http_req_duration(comp["archivos_caliente"], comp["url_contiene"])
    frio, statuses_f = perf.cargar_http_req_duration(comp["archivos_frio"], comp["url_contiene"])

    t_stat, t_p = __import__("scipy.stats", fromlist=["ttest_ind"]).ttest_ind(caliente, frio, equal_var=False)
    d = perf.cohen_d(caliente, frio)
    u_stat, u_p = __import__("scipy.stats", fromlist=["mannwhitneyu"]).mannwhitneyu(caliente, frio, alternative="two-sided")
    a12 = perf.vargha_delaney_a12(caliente, frio)

    resultados.append({
        "comparacion": comp, "desc_caliente": perf.descriptivos(caliente, statuses_c),
        "desc_frio": perf.descriptivos(frio, statuses_f), "welch_t": t_stat, "welch_p": t_p,
        "cohen_d": d, "mw_u": u_stat, "mw_p": u_p, "a12": a12,
    })

p_ajustados = perf.holm_bonferroni([r["mw_p"] for r in resultados])

for r, p_adj in zip(resultados, p_ajustados):
    comp = r["comparacion"]
    print("=" * 78)
    print(comp["nombre"])
    print("=" * 78)
    perf.formatear_descriptivos("Caliente", r["desc_caliente"])
    perf.formatear_descriptivos("Frio", r["desc_frio"])
    print()
    print(f"  Welch t-test (contraste): t = {r['welch_t']:.3f}, p = {r['welch_p']:.6e}, d de Cohen = {r['cohen_d']:.4f}")
    print(f"  Mann-Whitney U (test correcto): U = {r['mw_u']:.1f}, p = {r['mw_p']:.6e}, "
          f"p ajustado Holm-Bonferroni = {p_adj:.6e}")
    print(f"  A12 Vargha-Delaney: {r['a12']:.4f} ({perf.interpretar_a12(r['a12'])})")
    print()


Catalogo (publico) -- caliente vs frio
  Caliente:
    n peticiones     : 4500
    media (ms)       : 21.3789
    mediana (ms)     : 13.0124
    desv. estandar   : 32.7063
    p90 / p95 / p99  : 31.7491 / 50.1748 / 205.5998
    tasa error >=500 : 0.0000%
  Frio:
    n peticiones     : 4500
    media (ms)       : 13.6241
    mediana (ms)     : 9.0400
    desv. estandar   : 14.5160
    p90 / p95 / p99  : 22.3932 / 39.1448 / 89.9758
    tasa error >=500 : 0.0000%

  Welch t-test (contraste): t = 14.538, p = 4.112476e-47, d de Cohen = 0.3065
  Mann-Whitney U (test correcto): U = 13841240.0, p = 9.677117e-200, p ajustado Holm-Bonferroni = 9.677117e-200
  A12 Vargha-Delaney: 0.6835 (efecto mediano)

Comisiones /api/v1/admin/reportes/finanzas (protegido) -- caliente vs frio
  Caliente:
    n peticiones     : 7500
    media (ms)       : 19.6563
    mediana (ms)     : 15.0873
    desv. estandar   : 18.1878
    p90 / p95 / p99  : 34.4955 / 45.0549 / 99.1680
    tasa error >=500 : 0.0000%
  Frio:

**Comparar contra:** `docs/mediciones/perf/REPORTE-PERF.md` y
`docs/mediciones/perf/salida-inferencial.txt` (generado por `make perf-stats`) — mismos NDJSON de
entrada, mismas funciones, deben coincidir.

## 3. SUS (System Usability Scale)

Reutiliza `docs/mediciones/sus/analisis-sus.py` (puntaje Brooke 1986, IC 95% paramétrico t-Student)
y `docs/mediciones/sus/bootstrap-sus.py` (IC 95% bootstrap percentil, 10.000 remuestreos, semilla
fija `20260904`), ambos importados como módulo — ninguna fórmula se reimplementa aquí.

In [3]:
sus = cargar_modulo(Path("sus/analisis-sus.py"), "analisis_sus")
boot = cargar_modulo(Path("sus/bootstrap-sus.py"), "bootstrap_sus")

ruta_csv = Path("sus/sus-raw.csv")
participantes = sus.cargar_respuestas(ruta_csv)
puntajes = [sus.puntaje_sus(p["respuestas"]) for p in participantes]

import statistics
n = len(participantes)
media = statistics.mean(puntajes)
mediana = statistics.median(puntajes)
dt = statistics.stdev(puntajes) if n > 1 else 0.0

print(f"n participantes: {n}")
print(f"Media: {media:.2f} | Mediana: {mediana:.2f} | Desv. tipica: {dt:.2f}")
print(f"SHA-256 de {ruta_csv.name}: {sus.sha256_archivo(ruta_csv)}")

if n > 1:
    df = n - 1
    t = sus.valor_critico_t(df)
    margen = t * (dt / (n ** 0.5))
    print(f"IC 95% parametrico (t de Student, df={df}): [{media - margen:.2f}, {media + margen:.2f}]")

print(f"Interpretacion (Bangor): {sus.interpretacion_bangor(media)}")
umbral = "SUPERA" if media > sus.UMBRAL_PROYECTO else "NO SUPERA"
print(f"Umbral del proyecto (>{sus.UMBRAL_PROYECTO}): {umbral}")

ic_inf, ic_sup = boot.bootstrap_percentil(puntajes, boot.N_REMUESTREOS, boot.SEMILLA)
print()
print(f"IC 95% bootstrap percentil ({boot.N_REMUESTREOS} remuestreos, semilla {boot.SEMILLA}): "
      f"[{ic_inf:.2f}, {ic_sup:.2f}]")


n participantes: 16
Media: 61.25 | Mediana: 66.25 | Desv. tipica: 22.08
SHA-256 de sus-raw.csv: 728e99c7881010791818a43e68155f96165325983acff5b1cc435afca59b715b
IC 95% parametrico (t de Student, df=15): [49.49, 73.01]
Interpretacion (Bangor): D
Umbral del proyecto (>68.0): NO SUPERA

IC 95% bootstrap percentil (10000 remuestreos, semilla 20260904): [50.47, 71.41]


**Comparar contra:** `docs/mediciones/sus/REPORTE-SUS.md`,
`docs/mediciones/sus/salida-sus.txt` y `salida-bootstrap-sus.txt` — mismo CSV, mismas funciones
(literalmente las mismas, importadas, no reescritas), deben coincidir dígito a dígito. El SHA-256
impreso debe coincidir con el citado en `DATA-PROVENANCE`.

## 4. Lighthouse (medias por ruta/perfil)

No existía un script Python previo para esta parte. Lee los JSON de la corrida final canónica
`lhci-20260904-1545-{mobile,desktop}-prod-*-run{1,2,3}.report.json` — identificada en
`docs/mediciones/lighthouse/REPORTE-LIGHTHOUSE.md` como "la que se cita como resultado final" — y
extrae los scores de categoría (`performance`, `accessibility`, `best-practices`, `seo`) de cada
run, más su media por ruta/perfil.

In [4]:
import json
import re
from collections import defaultdict
from pathlib import Path

LH_DIR = Path("lighthouse")
CORRIDA = "20260904-1545"
CATEGORIAS = ["performance", "accessibility", "best-practices", "seo"]

reportes = sorted(LH_DIR.glob(f"lhci-{CORRIDA}-*-prod-*-run*.report.json"))
if not reportes:
    raise FileNotFoundError(f"No se encontraron reportes de la corrida {CORRIDA} en {LH_DIR}")

patron = re.compile(rf"lhci-{CORRIDA}-(mobile|desktop)-prod-(.+)-run(\d+)\.report\.json$")

por_ruta_perfil = defaultdict(list)
for ruta_archivo in reportes:
    m = patron.search(ruta_archivo.name)
    if not m:
        continue
    perfil, ruta, run = m.group(1), m.group(2), int(m.group(3))
    with ruta_archivo.open(encoding="utf-8") as f:
        data = json.load(f)
    scores = {cat: round(data["categories"][cat]["score"] * 100) for cat in CATEGORIAS}
    por_ruta_perfil[(perfil, ruta)].append((run, scores))

print(f"{'Perfil':<9} {'Ruta':<22} {'Run':<4} " + " ".join(f"{c:<16}" for c in CATEGORIAS))
for (perfil, ruta), corridas in sorted(por_ruta_perfil.items()):
    corridas.sort(key=lambda x: x[0])
    for run, scores in corridas:
        fila = " ".join(f"{scores[c]:<16}" for c in CATEGORIAS)
        print(f"{perfil:<9} {ruta:<22} {run:<4} {fila}")
    medias = {c: sum(s[c] for _, s in corridas) / len(corridas) for c in CATEGORIAS}
    fila_media = " ".join(f"{medias[c]:<16.2f}" for c in CATEGORIAS)
    print(f"{perfil:<9} {ruta:<22} {'media':<4} {fila_media}")
    print()


Perfil    Ruta                   Run  performance      accessibility    best-practices   seo             
desktop   auth_login             1    99               93               96               100             
desktop   auth_login             2    100              93               96               100             
desktop   auth_login             3    100              93               96               100             
desktop   auth_login             media 99.67            93.00            96.00            100.00          

desktop   explorar               1    94               93               96               100             
desktop   explorar               2    95               93               96               100             
desktop   explorar               3    92               93               96               100             
desktop   explorar               media 93.67            93.00            96.00            100.00          

desktop   explorar_creadores     1    99  

**Comparar contra:** las tablas "Resultados — mobile" / "Resultados — desktop" de
`docs/mediciones/lighthouse/REPORTE-LIGHTHOUSE.md` (sección de la corrida `15:45`) — cada celda
`X / Y / Z` del reporte corresponde a los tres `run` impresos arriba para esa ruta/perfil.

## Conclusión

Las cuatro secciones anteriores, ejecutadas de arriba a abajo sobre una copia limpia del
repositorio (sin ejecutar de nuevo `k6`, Lighthouse ni JaCoCo), reproducen las cifras ya
publicadas en `REPORTE-PERF.md`, `REPORTE-SUS.md`, `REPORTE-LIGHTHOUSE.md` y el reporte de
cobertura, a partir únicamente de los artefactos crudos versionados en `docs/mediciones/`. Esto
satisface la regla de reproducibilidad de la guía (§4.1) y cierra OBS-R1-05.